# Addis Ababa Road Traffic Accident Analysis
## Stage 4: Exploratory Data Analysis & Visualizations (ALX Module 1)

**Project:** Understanding and Trusting Road Accident Data: An Exploratory Analysis of Addis Ababa  
**Author:** YonInsights (`yo2120018@gmail.com`)  
**Kernel:** Python (Addis Ababa Road Safety)

---

## Step 1: Bar Charts & Pie Charts with Matplotlib
**Research Question:** How are recorded road traffic collisions distributed across accident severity categories in Addis Ababa?

- **Data required:** Categorical target variable `Accident_severity`.
- **Why Bar Chart:** Best for comparing absolute volumes along a common baseline without perceptual distortion.
- **Why Donut/Pie Chart:** Useful for illustrating relative composition of the whole (proportions), but small slices (e.g. Fatal injury at 2.4%) require explicit percentage callouts.
- **Figure output:** Saved to `../reports/figures/01_severity_distribution.png`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# 1. Setup paths
DATA_PATH = Path("../Data/Addis_Ababa_city_RTA.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Data/Addis_Ababa_city_RTA.csv")

FIG_DIR = Path("../reports/figures")
if not FIG_DIR.exists():
    FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load dataset
df = pd.read_csv(DATA_PATH)

# 3. Compute counts and percentages for Accident_severity
severity_counts = df['Accident_severity'].value_counts()
categories = severity_counts.index.tolist()
counts = severity_counts.values
percentages = (counts / counts.sum()) * 100

# 4. Define palette: Muted Blue (Slight), Amber (Serious), Crimson (Fatal)
colors = ['#2b5c8f', '#e67e22', '#c0392b']

# 5. Create side-by-side comparison figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Figure 1: Distribution of Recorded Accident Severity in Addis Ababa', fontsize=14, fontweight='bold', y=1.02)

# Subplot 1: Bar Chart (Absolute Counts)
bars = ax1.bar(categories, counts, color=colors, edgecolor='black', linewidth=0.8, width=0.55)
ax1.set_title('Absolute Incident Counts', fontsize=12, pad=10)
ax1.set_ylabel('Number of Recorded Collisions', fontsize=11)
ax1.set_xlabel('Assessed Severity Level', fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.5)

for bar, count, pct in zip(bars, counts, percentages):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 150,
             f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax1.set_ylim(0, max(counts) * 1.15)

# Subplot 2: Donut / Pie Chart (Proportions)
explode = (0.02, 0.05, 0.12)  # Emphasize the fatal category slice
wedges, texts, autotexts = ax2.pie(
    counts,
    labels=categories,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    explode=explode,
    wedgeprops=dict(width=0.6, edgecolor='black', linewidth=0.8),
    pctdistance=0.75
)
for autotext in autotexts:
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')
    autotext.set_color('white')

ax2.set_title('Relative Proportions (%)', fontsize=12, pad=10)

plt.tight_layout()
save_path = FIG_DIR / "01_severity_distribution.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {save_path}")
plt.show()

---

## Step 2: DataFrame Preparation for Temporal Visualization
**Research Question:** At what hours of the day do recorded collisions peak in Addis Ababa, and does the proportion of severe or fatal injuries increase during nighttime hours?

### Data Preparation Rationale:
1. The raw `Time` column is formatted as strings (`HH:MM:SS`). We parse this column and extract the **`Hour` of the day** (`0` to `23`) using `pd.to_datetime().dt.hour`.
2. We cross-tabulate `Hour` against `Accident_severity` using `pd.crosstab()`.
3. We calculate the combined **Severe/Fatal Injury Rate (%)** for each hour: $$\text{Severe/Fatal Rate} = \frac{\text{Serious Injury} + \text{Fatal Injury}}{\text{Total Collisions at that Hour}} \times 100$$

In [ ]:
# 1. Feature Engineering: Extract integer Hour (0-23) from Time string
df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour

# 2. Compute hourly collision totals
hourly_counts = df.groupby('Hour').size()

# 3. Compute cross-tabulation between Hour and Severity
crosstab_hourly = pd.crosstab(df['Hour'], df['Accident_severity'])

# 4. Calculate Severe & Fatal combined rate
severe_fatal_pct = ((crosstab_hourly['Serious Injury'] + crosstab_hourly['Fatal injury']) / hourly_counts * 100).round(2)

# 5. Build structured analytical DataFrame for plotting
hourly_df = pd.DataFrame({
    'Total_Collisions': hourly_counts,
    'Slight_Injury_Count': crosstab_hourly['Slight Injury'],
    'Serious_Injury_Count': crosstab_hourly['Serious Injury'],
    'Fatal_Injury_Count': crosstab_hourly['Fatal injury'],
    'Severe_Fatal_Rate_Pct': severe_fatal_pct
})

print("=== Prepared Hourly Aggregation DataFrame (Sample Hours) ===")
hourly_df.loc[[0, 3, 7, 12, 17, 21]]

---

## Step 3: Interactive Visualization with Plotly

### Why Plotly for this Question?
- **Dual-Axis Integration:** We need to plot two completely different scales simultaneously: **Volume** (bars ranging from 100 to 1,300 crashes) and **Rate** (line ranging from 10% to 25%).
- **Interactivity:** Road safety officials need to hover over specific hours to inspect exact counts and fatality rates.
- **Reproducible Export:** Saved as an optimized standalone HTML visualization to `../reports/figures/02_hourly_severity_interactive.html`.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Initialize figure with dual y-axes
fig_hourly = make_subplots(specs=[[{"secondary_y": True}]])

# 2. Trace 1: Hourly Collision Volume (Bar Chart on Primary Y-Axis)
fig_hourly.add_trace(
    go.Bar(
        x=hourly_df.index,
        y=hourly_df['Total_Collisions'],
        name='Collision Volume',
        marker_color='#2b5c8f',
        opacity=0.85,
        hovertemplate='<b>Hour %{x}:00</b><br>Total Crashes: %{y:,}<extra></extra>'
    ),
    secondary_y=False
)

# 3. Trace 2: Severe/Fatal Injury Rate (Line + Markers on Secondary Y-Axis)
fig_hourly.add_trace(
    go.Scatter(
        x=hourly_df.index,
        y=hourly_df['Severe_Fatal_Rate_Pct'],
        name='Severe/Fatal Rate (%)',
        mode='lines+markers',
        line=dict(color='#c0392b', width=3),
        marker=dict(size=7, color='#c0392b'),
        hovertemplate='<b>Hour %{x}:00</b><br>Severe/Fatal Rate: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=True
)

# 4. Layout and formatting
fig_hourly.update_layout(
    title='<b>Figure 2: Hourly Collision Volume vs. Severe Injury Rate in Addis Ababa</b>',
    xaxis=dict(title='Hour of the Day (24-Hour Clock)', tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(title='Total Recorded Collisions', showgrid=True),
    yaxis2=dict(title='Severe or Fatal Injury Rate (%)', showgrid=False, range=[0, 30]),
    hovermode='x unified',
    template='plotly_white',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
    margin=dict(l=60, r=60, t=70, b=60)
)

# 5. Export interactive figure
interactive_out = FIG_DIR / "02_hourly_severity_interactive.html"
fig_hourly.write_html(str(interactive_out), include_plotlyjs='cdn')
print(f"Interactive Plotly figure exported to: {interactive_out.resolve()}")

# Display inline in Jupyter notebook
fig_hourly.show()